In [ ]:
# ============================================================
# DoubleML 因果效应分析（修正版 v2, 2026-10-15）
# 相对旧版的修正：
# 1) 处理变量重标定：Hours_Studied/5、Attendance/10 后再送入 DoubleML，
#    使报告的 ATE 直接对应 "+5 hours" / "+10 percentage points" 的干预增量，
#    与 Table 5 的 Intervention Increment 标签及 CATE 的 Δ 口径一致
#    （修复编辑意见 Issue 1：旧版把"单位系数"错误标注为"+5h/+10%"的增量效应）。
# 2) 数据划分统一为 test_size=0.2，与 edu_performance_predictor.ipynb
#    及 student_performance_dashboard.py 保持一致（旧版此处为 0.3）。
# 3) "Attendance +10%" 歧义修正为 "+10 percentage points"（Attendance 取值 60-100）。
# 4) 删除未使用的 ml_g/ml_m；森林图标注偏移改为相对坐标，避免数值改大后重叠。
# 5) DoubleML 折划分前固定 np.random.seed(42)，保证结果可复现。
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from doubleml import DoubleMLData, DoubleMLPLR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 600
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['legend.fontsize'] = 10
sns.set_style("whitegrid")

# ---------- 数据预处理（与 edu_performance_predictor.ipynb / dashboard 一致） ----------
df = pd.read_csv('StudentPerformanceFactors.csv')
df_clean = df[df['Exam_Score'] <= 100].copy()

for col in df_clean.columns:
    if df_clean[col].dtype == 'object':
        df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])
    else:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

ordinal_mappings = {
    'Motivation_Level': {'Low': 0, 'Medium': 1, 'High': 2},
    'Parental_Involvement': {'Low': 0, 'Medium': 1, 'High': 2},
    'Teacher_Quality': {'Low': 0, 'Medium': 1, 'High': 2},
    'Family_Income': {'Low': 0, 'Medium': 1, 'High': 2},
    'Distance_from_Home': {'Near': 0, 'Moderate': 1, 'Far': 2},
    'Access_to_Resources': {'Low': 0, 'Medium': 1, 'High': 2},
    'Peer_Influence': {'Negative': 0, 'Neutral': 1, 'Positive': 2},
    'Parental_Education_Level': {'High School': 0, 'College': 1, 'Postgraduate': 2},
}

for col, mapping in ordinal_mappings.items():
    df_clean[col] = df_clean[col].map(mapping)

numeric_cols = ['Hours_Studied', 'Attendance', 'Sleep_Hours', 'Previous_Scores',
                'Tutoring_Sessions', 'Physical_Activity']
cat_cols = ['School_Type', 'Gender', 'Internet_Access',
            'Extracurricular_Activities', 'Learning_Disabilities']

feature_cols = numeric_cols + list(ordinal_mappings.keys()) + cat_cols
X = df_clean[feature_cols].copy()
y = df_clean['Exam_Score'].copy()

X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=False)

# 【修正2】统一 test_size=0.2
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

print(f"数据维度: {X_encoded.shape}")
print(f"训练集: {X_train.shape}, 测试集: {X_test.shape}")

print("\n" + "="*60)
print("PART 1: DoubleML ATE 估计（4个关键变量，按干预增量口径）")
print("="*60)

# scale: 将处理变量除以 scale 后送入 DoubleML，
# 使系数直接对应 desc 所述增量（PLR 系数随 D 线性缩放，z 统计量不变）
key_treatments = {
    'Hours_Studied':        {'delta': 5,  'desc': '+5 hours',              'scale': 5},
    'Attendance':           {'delta': 10, 'desc': '+10 percentage points', 'scale': 10},
    'Access_to_Resources':  {'delta': 1,  'desc': '+1 level',              'scale': 1},
    'Parental_Involvement': {'delta': 1,  'desc': '+1 level',              'scale': 1},
}

ate_results = []
dml_full = pd.concat([X_encoded.reset_index(drop=True), y.reset_index(drop=True)], axis=1)

for treatment, info in key_treatments.items():
    if treatment not in X_encoded.columns:
        print(f"跳过 {treatment}（不在编码后特征中）")
        continue

    print(f"\n>>> 正在估计 {treatment} ({info['desc']}) 的 ATE...")

    # 【修正1】重标定处理变量，使 ATE = 干预增量对应的效应
    data_t = dml_full.copy()
    data_t[treatment] = data_t[treatment] / info['scale']

    dml_data = DoubleMLData(
        data=data_t,
        y_col='Exam_Score',
        d_cols=treatment,
        x_cols=[col for col in X_encoded.columns if col != treatment]
    )

    ml_g = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
    ml_m = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)

    # 【修正5】DoubleML 在构造对象时即随机划分 cross-fitting 折，
    # 必须在 DoubleMLPLR() 之前固定种子，否则同一管线两次运行结果不同
    # （这正是旧版 Table 5 与 Figure 19/dashboard 数值不一致的根源）
    np.random.seed(42)
    dml_plr = DoubleMLPLR(dml_data, ml_g, ml_m, n_folds=5)
    dml_plr.fit()

    ate_inc = dml_plr.coef[0]          # 每"增量单位"的效应
    ci = dml_plr.confint()
    ci_lower = ci.iloc[0, 0]
    ci_upper = ci.iloc[0, 1]

    ate_results.append({
        'Treatment': treatment,
        'Desc': info['desc'],
        'Delta': info['delta'],
        'ATE': ate_inc,                          # 增量口径：+5h / +10pp / +1 level
        'CI_Lower': ci_lower,
        'CI_Upper': ci_upper,
        'ATE_per_unit': ate_inc / info['scale']  # 单位口径：每1小时 / 每1个百分点 / 每1级
    })

    print(f"    ATE({info['desc']}) = {ate_inc:.4f} (95% CI: [{ci_lower:.4f}, {ci_upper:.4f}])"
          f"  |  per-unit = {ate_inc/info['scale']:.4f}")

ate_df = pd.DataFrame(ate_results)
ate_df = ate_df.sort_values('ATE', ascending=True).reset_index(drop=True)
print("\n=== ATE 汇总表（增量口径，与 CATE 的 Δ 一致） ===")
print(ate_df[['Treatment', 'Desc', 'ATE', 'CI_Lower', 'CI_Upper', 'ATE_per_unit']].round(4))

ate_df.to_csv('ate_results_4variables.csv', index=False)

# ---------- ATE 森林图 ----------
fig, ax = plt.subplots(figsize=(12, 7))

y_pos = np.arange(len(ate_df))
colors = ['#2E86AB' if row['ATE'] > 0 else '#A23B72' for _, row in ate_df.iterrows()]

for i, (_, row) in enumerate(ate_df.iterrows()):
    ax.plot([row['CI_Lower'], row['CI_Upper']], [i, i],
            color='#555555', linewidth=3, zorder=1, solid_capstyle='round')
    ax.plot(row['CI_Lower'], i, marker='o', markersize=6,
            color='#555555', zorder=2, fillstyle='none', markeredgewidth=2)
    ax.plot(row['CI_Upper'], i, marker='o', markersize=6,
            color='#555555', zorder=2, fillstyle='none', markeredgewidth=2)

ax.scatter(ate_df['ATE'], y_pos, color=colors, s=200, zorder=5,
           edgecolors='black', linewidth=2, marker='D')

ax.axvline(x=0, color='red', linestyle='--', linewidth=2, alpha=0.7, label='No Effect (ATE=0)')

treatment_labels = {
    'Hours_Studied': 'Hours Studied (+5 hours)',
    'Attendance': 'Attendance (+10 pp)',
    'Access_to_Resources': 'Access to Resources (+1 level)',
    'Parental_Involvement': 'Parental Involvement (+1 level)'
}
ax.set_yticks(y_pos)
ax.set_yticklabels([treatment_labels.get(t, t) for t in ate_df['Treatment']], fontsize=12)
ax.set_xlabel('ATE on Exam Score per Stated Intervention Increment', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(axis='x', alpha=0.3, linestyle=':')

# 【修正4】相对坐标偏移，避免大数值下文字重叠
all_ci = ate_df[['CI_Lower', 'CI_Upper']].values.flatten()
x_range = all_ci.max() - all_ci.min()
for i, (_, row) in enumerate(ate_df.iterrows()):
    ax.text(row['CI_Upper'] + 0.02 * x_range, i + 0.12, f"ATE = {row['ATE']:.3f}",
            va='bottom', fontsize=11, fontweight='bold', color=colors[i])
    ax.text(row['CI_Upper'] + 0.02 * x_range, i - 0.28,
            f"[{row['CI_Lower']:.3f}, {row['CI_Upper']:.3f}]",
            va='bottom', fontsize=9, color='#666666', style='italic')

x_min = min(all_ci.min() - 0.05 * x_range, -0.05)
x_max = all_ci.max() + 0.35 * x_range
ax.set_xlim(x_min, x_max)

plt.tight_layout()
plt.savefig('fig_ate_forest_plot_4vars.png', dpi=600, bbox_inches='tight', facecolor='white')
plt.show()
print("\n✅ ATE 森林图已保存: fig_ate_forest_plot_4vars.png")

print("\n" + "="*60)
print("PART 2: CATE 异质性分析（4个关键变量，反事实预测，Δ 与 ATE 口径一致）")
print("="*60)

print("\n训练 Gradient Boosting 基础模型...")
gb_base = GradientBoostingRegressor(
    n_estimators=300, max_depth=6, learning_rate=0.05, random_state=42
)
gb_base.fit(X_train, y_train)
base_r2 = gb_base.score(X_test, y_test)
print(f"基础模型测试集 R²: {base_r2:.4f}")

cate_all_results = {}
cate_summary = []

for treatment, info in key_treatments.items():
    if treatment not in X_test.columns:
        continue

    print(f"\n>>> 正在计算 {treatment} ({info['desc']}) 的 CATE...")

    delta = info['delta']
    base_pred = gb_base.predict(X_test)
    X_cf = X_test.copy()
    X_cf[treatment] = X_cf[treatment] + delta

    # 边界截断：有序变量最高 2 级；Attendance 上限 100；Hours_Studied 上限 168
    if treatment in ['Access_to_Resources', 'Parental_Involvement']:
        X_cf[treatment] = X_cf[treatment].clip(0, 2)
    if treatment == 'Attendance':
        X_cf[treatment] = X_cf[treatment].clip(0, 100)
    if treatment == 'Hours_Studied':
        X_cf[treatment] = X_cf[treatment].clip(0, 24*7)

    cf_pred = gb_base.predict(X_cf)
    benefit = cf_pred - base_pred

    cate_all_results[treatment] = benefit

    cate_summary.append({
        'Treatment': treatment,
        'Mean_Benefit': benefit.mean(),
        'Std_Benefit': benefit.std(),
        'Max_Benefit': benefit.max(),
        'Top10_Mean': np.sort(benefit)[-10:].mean(),
        'Delta': info['desc']
    })

    top10_idx = np.argsort(benefit)[-10:][::-1]
    top10_benefit = benefit[top10_idx]

    fig, ax = plt.subplots(figsize=(10, 6))
    colors_bar = plt.cm.plasma(np.linspace(0.3, 0.9, 10))
    bars = ax.barh(range(10), top10_benefit[::-1], color=colors_bar, edgecolor='black', linewidth=0.5)

    for i, (bar, val) in enumerate(zip(bars, top10_benefit[::-1])):
        ax.text(val + 0.05, bar.get_y() + bar.get_height()/2,
                f'{val:.2f}', va='center', fontsize=10, fontweight='bold')

    ax.set_yticks(range(10))
    ax.set_yticklabels([f'Student {idx}' for idx in top10_idx[::-1]])
    ax.set_xlabel(f'Predicted Score Gain from {info["desc"]}', fontsize=12, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)

    ax.axvline(x=benefit.mean(), color='red', linestyle='--', linewidth=1.5, alpha=0.7,
               label=f'Population Mean ({benefit.mean():.2f})')
    ax.legend(loc='lower right')

    plt.tight_layout()
    fname = f'fig_cate_top10_{treatment.lower()}.png'
    plt.savefig(fname, dpi=600, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f"    ✅ CATE Top10 图已保存: {fname}")

    top10_df = pd.DataFrame({
        'Rank': range(1, 11),
        'Student_Index': top10_idx,
        'Base_Score': base_pred[top10_idx].round(2),
        'Counterfactual_Score': cf_pred[top10_idx].round(2),
        'Benefit': top10_benefit.round(2)
    })
    print(f"\n    Top 10 受益学生 ({treatment}):")
    print(top10_df.to_string(index=False))

cate_summary_df = pd.DataFrame(cate_summary)
print("\n=== CATE 异质性汇总表 ===")
print(cate_summary_df.round(4))
cate_summary_df.to_csv('cate_summary_4variables.csv', index=False)

# ---------- CATE 分布箱线图 ----------
fig, ax = plt.subplots(figsize=(12, 7))

cate_data_plot = []
labels_plot = []
for treatment, benefit in cate_all_results.items():
    cate_data_plot.append(benefit)
    labels_plot.append(treatment_labels.get(treatment, treatment))

bp = ax.boxplot(cate_data_plot, labels=labels_plot, patch_artist=True,
                notch=True, showfliers=False, widths=0.6)

colors_box = plt.cm.Set3(np.linspace(0, 1, len(bp['boxes'])))
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

means = [np.mean(b) for b in cate_data_plot]
ax.scatter(range(1, len(means)+1), means, color="red", s=100, zorder=5,
           marker="D", label="Mean CATE", edgecolors="black", linewidth=1)

for i, (treatment, benefit) in enumerate(cate_all_results.items()):
    mean_val = np.mean(benefit)
    std_val = np.std(benefit)
    q3 = np.percentile(benefit, 75)
    ax.text(i+1, q3 + 0.1, f"μ={mean_val:.2f}\nσ={std_val:.2f}",
            ha="center", va="bottom", fontsize=9, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.2", facecolor="yellow", edgecolor="black", alpha=0.7))

ax.set_ylabel('Predicted Treatment Effect (Score Points)', fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
ax.legend(loc='upper right')

plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig('fig_cate_distribution_4vars.png', dpi=600, bbox_inches='tight', facecolor='white')
plt.show()
print("\n✅ CATE 分布汇总图已保存: fig_cate_distribution_4vars.png")

# ---------- 联合结果 + 一致性校验 ----------
joint_table = ate_df.merge(
    cate_summary_df[['Treatment', 'Mean_Benefit', 'Std_Benefit', 'Max_Benefit', 'Top10_Mean']],
    on='Treatment'
)

print("\n联合结果（ATE 增量口径 + CATE 异质性，两者 Δ 一致）:")
print(joint_table[['Treatment', 'Desc', 'ATE', 'CI_Lower', 'CI_Upper',
                   'ATE_per_unit', 'Mean_Benefit', 'Std_Benefit',
                   'Max_Benefit', 'Top10_Mean']].round(4))
joint_table.to_csv('joint_ate_cate_results.csv', index=False)

print("\n=== 一致性校验：DoubleML ATE(增量) vs GB-CATE 均值 ===")
for _, row in joint_table.iterrows():
    ratio = row['Mean_Benefit'] / row['ATE'] if row['ATE'] != 0 else np.nan
    print(f"  {row['Treatment']:<22} ATE={row['ATE']:.3f}  CATE_mean={row['Mean_Benefit']:.3f}  ratio={ratio:.2f}")
print("说明：连续处理（Hours/Attendance）两者应接近；有序处理（Resources/Involvement）")
print("CATE 均值偏低属正常——已是 High 的学生触及上限截断，反事实增益为 0，")
print("而 PLR 假设恒定的每级效应。论文中需就此作出说明。")

# ---------- PART 3: 为 Dashboard 计算全部 7 个处理变量的统一预计算值 ----------
print("\n" + "="*60)
print("PART 3: Dashboard 预计算值（统一管线重算，修复硬编码不一致）")
print("="*60)

dashboard_treatments = {
    'Hours_Studied':        {'scale': 5,  'unit': 'per +5 hours/week'},
    'Attendance':           {'scale': 10, 'unit': 'per +10 percentage points'},
    'Previous_Scores':      {'scale': 1,  'unit': 'per point'},
    'Parental_Involvement': {'scale': 1,  'unit': 'per level'},
    'Sleep_Hours':          {'scale': 1,  'unit': 'per hour/night'},
    'Tutoring_Sessions':    {'scale': 1,  'unit': 'per session/week'},
    'Teacher_Quality':      {'scale': 1,  'unit': 'per level'},
}

precomputed = {}
for treatment, info in dashboard_treatments.items():
    if treatment not in X_encoded.columns:
        continue
    print(f">>> {treatment} ...")
    data_t = dml_full.copy()
    data_t[treatment] = data_t[treatment] / info['scale']
    dml_data = DoubleMLData(
        data=data_t, y_col='Exam_Score', d_cols=treatment,
        x_cols=[col for col in X_encoded.columns if col != treatment]
    )
    ml_g = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
    ml_m = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
    # 【修正5】DoubleML 在构造对象时即随机划分 cross-fitting 折，
    # 必须在 DoubleMLPLR() 之前固定种子，否则同一管线两次运行结果不同
    # （这正是旧版 Table 5 与 Figure 19/dashboard 数值不一致的根源）
    np.random.seed(42)
    dml_plr = DoubleMLPLR(dml_data, ml_g, ml_m, n_folds=5)
    dml_plr.fit()
    ci = dml_plr.confint()
    precomputed[treatment] = {
        'ate': round(float(dml_plr.coef[0]), 3),
        'ci_lower': round(float(ci.iloc[0, 0]), 3),
        'ci_upper': round(float(ci.iloc[0, 1]), 3),
        'unit': info['unit']
    }
    print(f"    {precomputed[treatment]}")

with open('dashboard_precomputed.json', 'w') as f:
    json.dump(precomputed, f, indent=2)
print("\n✅ Dashboard 预计算值已保存: dashboard_precomputed.json")



